# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [3]:
import duckdb
from pathlib import Path
import pandas as pd

# Verbindung zur DuckDB (Dateibasierte DB im Projektordner)
db_path = Path("data/processed/olist.duckdb")
# Verzeichnis anlegen, falls nicht vorhanden
db_path.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(db_path))

# Hilfsfunktion für SQL-Abfragen
def sql(q):
    return con.sql(q).df()

# Projektpfad bestimmen (Notebook liegt in /notebooks)
DATA = Path("../data/raw/brazilian-ecommerce")

In [4]:
# Alle CSV-Dateien in DuckDB registrieren
for f in DATA.glob("*.csv"):
    name = f.stem.replace("olist_", "").replace("_dataset", "")
    
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('{f.as_posix()}')
        """
           )
           
# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [5]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    Count(oi.product_id) AS product_count,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
GROUP BY c.customer_id, o.order_id, o.order_purchase_timestamp, 
                 o.order_approved_at, oi.price, oi.freight_value, 
                 c.customer_unique_id, c.customer_city, c.customer_state, 
                 c.customer_zip_code_prefix, o.order_status
    """)

In [6]:
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
0,3b6828a50ffe546942b7a473d70ac0fc,ccafc1c3f270410521c3c6f3b249870f,goiania,GO,74820,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,132.40,14.05,1
1,fc849c5fbc7e2b0c7cfebc138886c693,5bbc03d010ac4dbcfdfa2018349c8f6e,tocos,RJ,28148,210e60465099814a1d2c7635e4065153,delivered,2018-05-09 18:27:00,2018-05-09 18:54:13,199.00,32.79,1
2,d0e71d7a6dbca8ac513ac63d08a81fa0,5fa1f56316f4154109ceb42651385b7c,sao caetano do sul,SP,09521,5164933efe0cd14f31424b9badf81f19,delivered,2018-04-25 22:18:10,2018-04-25 22:31:56,129.90,12.00,1
3,45c9fcdb0cd88a5630cb4405c4d2e4a7,c6ff1299ecfd927f41df559f9b61e94a,biritiba-mirim,SP,08940,93a21c10557cc1b1ecfe2c3a808c648a,delivered,2018-02-27 10:05:56,2018-02-27 10:15:33,320.00,18.00,1
4,d1b370c90ad7dacb840010a6e89c7e89,29fbb2801362f8857a80e4c1a0c74aa5,belo horizonte,MG,31710,7835af1856de332f2f3c9204b740a3a2,delivered,2017-12-24 17:34:12,2017-12-24 17:48:13,49.99,14.10,1
...,...,...,...,...,...,...,...,...,...,...,...,...
101565,de1c64a7f4179d4216ee09d181aa2e6a,dde8dd36a5290a1bfb650e1c8037b88b,brasilia,DF,70340,d181dfce5d8355bc51c2370c6910d4bc,delivered,2017-07-06 17:25:17,2017-07-07 17:30:19,238.81,19.63,1
101566,7429462259e9d39e60127253ab7bdbe7,5bb8cdfb477b265ee4a460fcdfa5d0e4,praia grande,SP,11719,3a650c50958d54a9beebc3db169500cd,delivered,2018-02-13 23:19:30,2018-02-13 23:30:32,29.45,9.34,1
101567,e5426fc9fcee3dfa52fb048f6d0856e9,770c46cc8437a67264254f23c1786b97,campinas,SP,13084,6dce05516fc9a1e16b7a70e49ab656c2,delivered,2017-05-14 11:05:08,2017-05-14 11:15:13,26.14,9.36,1
101568,fbfb33cc34116ccf95b043f7cd31c692,1d7c9872acb163f50b958fa4b729dd02,santo antonio de padua,RJ,28470,4294d6c2b09c5f2438c01b11c91ff7a8,delivered,2018-06-10 13:42:58,2018-06-10 13:55:13,24.00,37.06,1


In [7]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04.180673,2018-01-01 12:06:22.988666,124.922151,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48.500000,40.800000,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20.500000,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31.500000,139.530000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,20.000000
std,NaN,NaN,189.479405,15.901388,0.474137


#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [8]:
df_rfm_eda.dtypes

customer_id                         object
customer_unique_id                  object
customer_city                       object
customer_state                      object
customer_zip_code_prefix            object
order_id                            object
order_status                        object
order_purchase_timestamp    datetime64[us]
order_approved_at           datetime64[us]
price                              float64
freight_value                      float64
product_count                        int64
dtype: object

In [9]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_count': 'int16', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

customer_id                      category
customer_unique_id               category
customer_city                    category
customer_state                   category
customer_zip_code_prefix         category
order_id                         category
order_status                     category
order_purchase_timestamp    datetime64[s]
order_approved_at           datetime64[s]
price                             float32
freight_value                     float32
product_count                       int16
dtype: object

In [10]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04,2018-01-01 12:06:22,124.922150,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48,40.799999,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31,139.529995,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,189.479401,15.901388,0.474137


In [11]:
df_rfm_eda.isna().sum()

customer_id                  0
customer_unique_id           0
customer_city                0
customer_state               0
customer_zip_code_prefix     0
order_id                     0
order_status                 0
order_purchase_timestamp     0
order_approved_at           14
price                        0
freight_value                0
product_count                0
dtype: int64

In [12]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
2610,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,59.900002,17.160000,1
5350,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,149.800003,13.630000,1
22495,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,309.899994,39.110001,1
22824,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,39.990002,14.520000,1
23456,f67cd1a215aae2a1074638bbd35a223a,bc1896dc77f49e6dec880445a9b443a3,rio de janeiro,RJ,21020,88083e8f64d95b932164187484d90212,delivered,2017-02-18 22:49:19,NaT,49.000000,14.520000,2
32166,29c35fc91fc13fb5073c8f30505d860d,7e1a5ca61b572d76b64b6688b9f96473,caninde,CE,62700,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaT,79.989998,26.820000,1
41141,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,79.989998,26.820000,1
45247,d5de688c321096d15508faae67a27051,d49f3dae6bad25d05160fc17aca5942d,conselheiro lafaiete,MG,36400,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,45.900002,14.520000,1
48030,1e101e0daffaddce8159d25a8e53f2b2,c8822fce1d0bfa7ddf0da24fff947172,macae,RJ,27945,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,79.989998,15.770000,1
75405,0bf35cac6cc7327065da879e2d90fae8,c4c0011e639bdbcf26059ddc38bd3c18,varzea paulista,SP,13225,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,28.990000,10.960000,1


In [13]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

order_approved_at,False,True
order_status,,
approved,2,0
canceled,464,0
delivered,99341,14
invoiced,319,0
processing,304,0
shipped,1119,0
unavailable,7,0


In [14]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count


In [15]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

Gesamte Duplikate: 0


In [16]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,0,1,0,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,0,1,0,0,0,0
000229ec398224ef6ca0657da4fc703e,0,0,1,0,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,0,1,0,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...
fffc94f6ce00a00581880bf54a75a037,0,0,1,0,0,0,0
fffcd46ef2263f404302a634eb57f7eb,0,0,1,0,0,0,0
fffce4705a9662cd70adb13d4a31832d,0,0,1,0,0,0,0


In [17]:
test['sum_status']=test.sum(axis=1)

In [18]:
test.loc[test['sum_status']!=1, :] 

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
002f98c0f7efd42638ed6100ca699b42,0,0,2,0,0,0,0,2
005d9a5423d47281ac463a968b3936fb,0,0,2,0,0,0,0,2
00946f674d880be1f188abc10ad7cf46,0,0,2,0,0,0,0,2
0097f0545a302aafa32782f1734ff71c,0,0,2,0,0,0,0,2
00bcee890eba57a9767c7b5ca12d3a1b,0,0,2,0,0,0,0,2
...,...,...,...,...,...,...,...,...
ffb18bf111fa70edf316eb0390427986,0,0,2,0,0,0,0,2
ffb8f7de8940249a3221252818937ecb,0,0,3,0,0,0,0,3
ffb9a9cd00c74c11c24aa30b3d78e03b,0,0,3,0,0,0,0,3


####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [19]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [20]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
ca3625898fbd48669d50701aba51cd5f,0,0,7,0,0,0,0,7
cf5c8d9f52807cb2d2f0a0ff54c478da,0,0,6,0,0,0,0,6
5a3b1c29a49756e75f1ef513383c0c12,0,0,6,0,0,0,0,6
b436eb981676e54c0bc9bcade0e079c4,0,0,5,0,0,0,0,5
bb82809ea3ca9f3edbe589b60e14e0cb,0,0,5,0,0,0,0,5
...,...,...,...,...,...,...,...,...
59b67c775c6a905fc4faac69ca74b5cb,0,0,2,0,0,0,0,2
59bccab4e9193a9229f7d1b73fcb47c3,0,0,2,0,0,0,0,2
59c0ed646a3b30d4054298988188486f,0,0,2,0,0,0,0,2


## Filterung nach der Bestellung mit den meisten Duplikaten

In [21]:
order_id = 'ca3625898fbd48669d50701aba51cd5f'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
2925,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,309.000000,1.84,1
22128,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,95.900002,0.15,2
29913,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,159.000000,3.67,2
47349,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,33.900002,1.84,1
48971,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,63.700001,0.15,1
69628,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,109.900002,0.15,1
77295,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,56.000000,3.68,2


In [22]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101099,101099,101099.000000,101099.000000,101099.000000
mean,2018-01-01 04:49:34,2018-01-01 15:09:29,124.649025,20.140503,1.108824
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 11:58:02,2017-09-13 20:10:21,40.799999,13.180000,1.000000
50%,2018-01-19 16:33:57,2018-01-20 09:08:37,79.000000,16.350000,1.000000
75%,2018-05-05 07:49:38,2018-05-05 14:13:51,139.000000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,188.526642,15.891350,0.473023


In [23]:
con.execute("CREATE TABLE customer_rfm AS SELECT * FROM df_rfm_eda")

### EDA für zweite Kernaufgabe

In [24]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    p.product_id,
    pcnt.product_category_name_english,
    Count(o.order_id) AS order_count,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    r.review_score
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id 
JOIN orders o ON oi.order_id = o.order_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
GROUP BY p.product_id, pcnt.product_category_name_english, o.order_status, o.order_purchase_timestamp,
         o.order_approved_at, oi.price, oi.freight_value, r.review_score
    """)

In [25]:
df_pc_eda

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
0,1121a43969b79e65010c2a5cdd0dcb6c,housewares,1,delivered,2018-04-01 02:36:48,2018-04-01 03:27:28,117.80,41.30,4
1,6b120063aeece762b2ef81da00c75ab9,toys,1,delivered,2017-09-09 15:17:43,2017-09-12 04:55:06,94.00,35.98,5
2,96592c73e37eabf051bfc0d1ab2bece1,furniture_decor,1,delivered,2018-07-16 10:47:55,2018-07-17 04:31:15,179.90,47.92,3
3,3b0f7951038b105522c2d566b54421f7,telephony,1,delivered,2018-03-02 15:06:22,2018-03-03 15:09:00,29.98,15.10,5
4,d4d998605a20a7c575f53f6c8be0cde5,baby,1,delivered,2018-02-27 19:14:20,2018-02-27 19:30:22,2288.00,96.50,5
...,...,...,...,...,...,...,...,...,...
100703,55782ee5db5083e1e3bb2dbb054fdf49,cool_stuff,1,delivered,2018-02-17 23:26:19,2018-02-18 00:26:38,215.00,25.96,<NA>
100704,f5889057b0c061390ce20f4b7b842ac0,bed_bath_table,1,delivered,2018-06-26 23:44:15,2018-06-27 01:35:26,49.90,22.28,<NA>
100705,d678178aa4291cd25a755a90188375c8,furniture_decor,2,delivered,2017-11-25 01:06:42,2017-11-25 03:52:51,32.99,16.11,<NA>
100706,c3f6113d5b61bc95468432072b27e23d,computers_accessories,1,delivered,2017-07-24 20:56:42,2017-07-24 21:05:21,18.90,15.10,<NA>


In [26]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
count,100708.000000,100708,100695,100708.000000,100708.000000,99940.0
mean,1.103597,2018-01-01 16:12:59.141210,2018-01-02 03:32:58.710333,124.151016,20.142170,4.088883
min,1.000000,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.0
25%,1.000000,2017-09-13 17:09:08,2017-09-14 02:45:40,40.140000,13.180000,4.0
50%,1.000000,2018-01-20 13:59:55.500000,2018-01-20 20:00:10,78.000000,16.360000,5.0
75%,1.000000,2018-05-05 21:22:12.750000,2018-05-06 13:50:11,139.000000,21.260000,5.0
max,20.000000,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,5.0
std,0.461837,NaN,NaN,187.484910,15.898289,1.342309


In [27]:
df_pc_eda.dtypes

product_id                               object
product_category_name_english            object
order_count                               int64
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
price                                   float64
freight_value                           float64
review_score                              Int64
dtype: object

In [28]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'product_id': 'category',
              'product_category_name_english': 'category', 
              'order_count': 'Int16', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

product_id                            category
product_category_name_english         category
order_count                              Int16
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
price                                  float32
freight_value                          float32
review_score                          category
dtype: object

In [29]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value
count,100708.0,100708,100695,100708.000000,100708.000000
mean,1.103597,2018-01-01 16:12:59,2018-01-02 03:32:58,124.151009,20.142170
min,1.0,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000
25%,1.0,2017-09-13 17:09:08,2017-09-14 02:45:40,40.139999,13.180000
50%,1.0,2018-01-20 13:59:55,2018-01-20 20:00:10,78.000000,16.360001
75%,1.0,2018-05-05 21:22:12,2018-05-06 13:50:11,139.000000,21.260000
max,20.0,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993
std,0.461837,NaN,NaN,187.484909,15.898289


In [30]:
df_pc_eda.isna().sum()

product_id                         0
product_category_name_english      0
order_count                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 13
price                              0
freight_value                      0
review_score                     768
dtype: int64

In [31]:
df_pc_eda['order_approved_at'] = df_pc_eda['order_approved_at'].fillna(
    pd.to_datetime(df_pc_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_pc_eda[df_pc_eda.isna().any(axis=1)].head(15)

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
1508,91dbbc0bb2edadd00ccfd122d2212760,toys,1,delivered,2017-12-11 01:00:21,2017-12-11 01:10:48,199.990005,20.889999,NaN
1509,0983cd4a5cabf1099659ce461511963c,health_beauty,1,delivered,2018-04-24 10:19:45,2018-04-24 18:29:37,99.989998,52.830002,NaN
1510,9cfb6da38dab3ad1f5e8cf3189ab6ae1,furniture_decor,1,delivered,2017-02-15 09:35:30,2017-02-15 09:45:10,74.900002,11.910000,NaN
1511,386eebb43722ab502f04f7900bd2451b,health_beauty,1,delivered,2018-02-19 08:23:36,2018-02-19 08:35:25,57.900002,11.880000,NaN
1512,f71c0f4ee48321ba6b82205b2fc9e08f,furniture_decor,1,delivered,2017-08-31 17:56:08,2017-08-31 18:05:23,98.900002,16.129999,NaN
1513,5d7c23067ed3fc8c6e699b9373d5890b,fashion_bags_accessories,1,delivered,2017-06-19 22:37:10,2017-06-21 02:30:29,49.000000,7.780000,NaN
1514,e03071a2d2410c9ef2be47b508cac95f,housewares,1,delivered,2017-03-13 10:26:02,2017-03-13 10:26:02,165.399994,19.440001,NaN
1515,15348d4e550d520b418c581a2012c0d9,home_construction,3,delivered,2017-11-26 19:11:20,2017-11-26 19:19:22,37.500000,17.600000,NaN
1516,10717ff440b2320081989126e858b220,bed_bath_table,1,delivered,2018-01-19 23:09:18,2018-01-19 23:18:27,138.000000,12.170000,NaN
1517,4fe644d766c7566dbc46fb851363cb3b,art,1,delivered,2018-04-05 01:30:10,2018-04-05 01:47:33,119.000000,12.170000,NaN


In [32]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

Gesamte Duplikate: 0


In [33]:
con.execute("CREATE TABLE product_category AS SELECT * FROM df_pc_eda")

### EDA für dritte Kernaufgabe

In [34]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
        r.review_id,
        r.review_score,
        r.review_comment_title,
        r.review_comment_message,
        r.review_creation_date,
        r.review_answer_timestamp,
    oi.product_id,
    pcnt.product_category_name_english,
    Count(oi.product_id) AS product_count,                
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
WHERE o.order_status IN ('delivered')
Group BY o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at,
         o.order_delivered_carrier_date, o.order_delivered_customer_date, o.order_estimated_delivery_date,
         r.review_id, r.review_score, r.review_comment_title, r.review_comment_message, 
                     r.review_creation_date, r.review_answer_timestamp,
         oi.product_id, s.seller_id, s.seller_city, s.seller_state, 
                     c.customer_city, c.customer_state, pcnt.product_category_name_english
    """)

In [35]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
0,bce1e6fae8db0875cde2048ee4220438,delivered,2017-12-28 10:45:14,2017-12-28 10:54:22,2017-12-28 22:04:14,2018-01-08 17:00:02,2018-01-29,0f8fd34e53f1e578d63523da9b9c811f,5,None,...,2018-01-09 20:00:27,c5b72065154ec27c2d1ed8a654c3348f,watches_gifts,1,b33e7c55446eabf8fe1a42d037ac7d6d,pradopolis,SP,uberlandia,MG,watches_gifts
1,a1f5afc5fe7a4ee4bef93aab1e95772f,delivered,2017-05-06 11:09:15,2017-05-11 23:31:25,2017-05-09 08:48:36,2017-05-16 08:05:15,2017-05-29,9f6f9a9f0ea344f6c6d5fa8adbcca548,5,None,...,2017-05-19 18:03:15,f4f67ccaece962d013a4e1d7dc3a61f7,auto,1,8581055ce74af1daba164fdbd55a40de,guarulhos,SP,sao bernardo do campo,SP,auto
2,8dca52c1ec5a814dc9ebd26524ae8b3e,delivered,2017-11-08 20:41:49,2017-11-10 12:28:15,2017-11-10 19:48:34,2017-11-17 20:15:28,2017-12-05,8c26c38cf5e2dd8fe99a6f812634a845,5,None,...,2017-11-19 11:12:21,726e355b119d1f8186283de59fc5b702,luggage_accessories,1,dbc22125167c298ef99da25668e1011f,borda da mata,MG,itatiaia,RJ,luggage_accessories
3,d6021b689d20a82e885a61d75888df6a,delivered,2018-04-03 07:28:21,2018-04-03 07:50:12,2018-04-05 14:19:22,2018-04-09 21:08:38,2018-04-25,259d7e24fdc165eca679122592e62d41,3,None,...,2018-04-11 10:16:52,4f9172df8e9ae60aa90c14fb36afbcbb,musical_instruments,2,0e982cff76cc0579f632cea8a0e38c9d,itajai,SC,votorantim,SP,musical_instruments
4,eef5a4bd5dd37ec132d383869df85eb1,delivered,2017-08-02 23:08:01,2017-08-03 20:03:20,2017-08-04 16:06:00,2017-08-10 20:42:15,2017-08-24,2bd75265617ceff1c8746f58784307ab,4,None,...,2017-08-12 06:33:34,cc2232dbef2c9fca23f4c7f6a19a42e3,small_appliances,1,198c7ea11960a9844b544d9bcdca860c,jacutinga,MG,nova iguacu,RJ,small_appliances
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98660,1b7efcda93ef629e74df8b1fcd2b1f92,delivered,2017-09-05 08:13:17,2017-09-05 08:25:12,2017-09-05 22:07:47,2017-09-15 18:22:22,2017-10-02,06d87054d145c4352c569d42fe1f01d2,1,None,...,2017-09-18 17:57:02,b5e13c9a353102f79c6206ff5cb61a50,toys,1,a49928bcdf77c55c6d6e05e09a9b4ca5,sao paulo,SP,eunapolis,BA,toys
98661,415f2ac1a1d977ad1f20427eec4c62ce,delivered,2017-04-24 19:58:33,2017-04-26 09:36:17,2017-04-27 06:18:09,2017-05-06 11:42:42,2017-05-15,de6978c43d5b433b38b72e90872cf8ca,5,None,...,2017-05-07 21:54:30,122f396fa6d0f9070c6e721f8d833d2b,sports_leisure,1,50c9975695009e5e6473912e83a6d1da,claudio,MG,bebedouro,SP,sports_leisure
98662,a8fee5f264541f904519912b305d0bf6,delivered,2018-04-23 09:31:12,2018-04-24 17:29:46,2018-04-24 00:16:47,2018-04-26 16:56:50,2018-05-16,10652d90aed6e45f47a182b96c3d2e6f,1,None,...,2018-04-27 20:54:29,c9c6fde711572c1ad99ca12728c6af00,telephony,1,562fc2f2c2863ab7e79a9e4388a58a14,campinas,SP,mongagua,SP,telephony
98663,9bafbb16dccffbb09e9d88acb1a932f4,delivered,2017-11-17 13:10:42,2017-11-18 02:31:07,2017-11-21 20:53:52,2017-11-22 20:59:32,2017-11-30,90f54a3c2fe89072db8e2cb8b5ad128f,4,None,...,2017-11-25 15:34:26,8a5835aeef83efa6aa947f84f92deb0e,watches_gifts,1,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,sao paulo,SP,watches_gifts


In [36]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98665,98652,98663,98657,98665,98665.000000,98665,98665,98665.000000
mean,2018-01-02 11:40:02.847707,2018-01-02 22:58:48.161487,2018-01-05 16:48:14.909003,2018-01-14 22:24:39.031310,2018-01-26 06:41:08.352505,4.127644,2018-01-14 17:11:58.613489,2018-01-17 20:45:37.395662,1.099255
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000,2016-10-06 00:00:00,2016-10-07 18:32:28,1.000000
25%,2017-09-14 13:57:21,2017-09-14 22:25:18.250000,2017-09-18 19:05:10,2017-09-26 16:48:05,2017-10-05 00:00:00,4.000000,2017-09-27 00:00:00,2017-09-29 12:12:07,1.000000
50%,2018-01-21 13:28:36,2018-01-22 14:03:06,2018-01-24 18:52:39,2018-02-02 21:33:30,2018-02-16 00:00:00,5.000000,2018-02-03 00:00:00,2018-02-06 21:05:12,1.000000
75%,2018-05-06 18:49:55,2018-05-07 16:55:38.250000,2018-05-09 10:13:00,2018-05-16 17:24:34,2018-05-28 00:00:00,5.000000,2018-05-17 00:00:00,2018-05-20 20:13:32,1.000000
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000,2018-08-31 00:00:00,2018-10-29 12:27:35,20.000000
std,NaN,NaN,NaN,NaN,NaN,1.308843,NaN,NaN,0.451793


In [37]:
df_service_eda.dtypes


order_id                                   object
order_status                               object
order_purchase_timestamp           datetime64[us]
order_approved_at                  datetime64[us]
order_delivered_carrier_date       datetime64[us]
order_delivered_customer_date      datetime64[us]
order_estimated_delivery_date      datetime64[us]
review_id                                  object
review_score                                int64
review_comment_title                       object
review_comment_message                     object
review_creation_date               datetime64[us]
review_answer_timestamp            datetime64[us]
product_id                                 object
product_category_name_english              object
product_count                               int64
seller_id                                  object
seller_city                                object
seller_state                               object
customer_city                              object


In [38]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category',
              'order_status': 'category',
              'order_approved_at': 'datetime64[s]',
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_delivered_carrier_date': 'datetime64[s]', 
              'order_delivered_customer_date': 'datetime64[s]',
              'order_estimated_delivery_date': 'datetime64[s]',
              'review_id': 'category',
              'review_score': 'Int16',
              'review_comment_title': 'category',
              'review_comment_message': 'category',
              'review_creation_date': 'datetime64[s]',
              'review_answer_timestamp': 'datetime64[s]',
              'product_id': 'category',
              'product_category_name_english': 'category',
              'product_count': 'Int16',
              'seller_id': 'category',
              'seller_city': 'category',
              'seller_state': 'category',
              'customer_city': 'category',
              'customer_state': 'category', 
              
              }
df_service_eda = df_service_eda.astype(col_dtypes)
df_service_eda.dtypes

order_id                                category
order_status                            category
order_purchase_timestamp           datetime64[s]
order_approved_at                  datetime64[s]
order_delivered_carrier_date       datetime64[s]
order_delivered_customer_date      datetime64[s]
order_estimated_delivery_date      datetime64[s]
review_id                               category
review_score                               Int16
review_comment_title                    category
review_comment_message                  category
review_creation_date               datetime64[s]
review_answer_timestamp            datetime64[s]
product_id                              category
product_category_name_english           category
product_count                              Int16
seller_id                               category
seller_city                             category
seller_state                            category
customer_city                           category
customer_state      

In [39]:
df_service_eda.isna().sum()

order_id                               0
order_status                           0
order_purchase_timestamp               0
order_approved_at                     13
order_delivered_carrier_date           2
order_delivered_customer_date          8
order_estimated_delivery_date          0
review_id                              0
review_score                           0
review_comment_title               86959
review_comment_message             58096
review_creation_date                   0
review_answer_timestamp                0
product_id                             0
product_category_name_english          0
product_count                          0
seller_id                              0
seller_city                            0
seller_state                           0
customer_city                          0
customer_state                         0
product_category_name_english_1        0
dtype: int64

In [40]:
df_service_eda['order_approved_at'] = df_service_eda['order_approved_at'].fillna(
    pd.to_datetime(df_service_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
1345,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30,bb311d9562ecbefc8e4be756d8999892,5,NaN,...,2018-07-10 11:38:13,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,1,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,sumare,SP,watches_gifts
7897,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16,c0dd6bec0375c376f044af102118526f,5,Entrega super rápida.,...,2018-06-29 16:26:37,2167c8f6252667c0eb9edd51520706a1,industry_commerce_and_business,1,0bb738e4d789e63e2267697c42d35a2d,sao roque,SP,quadra,SP,industry_commerce_and_business
18297,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26,0d4c56af896dd6eb9de8edbaa1902d22,1,Péssimo,...,2018-06-16 13:55:00,a2a7efc985315e86d4f0f705701b342b,computers_accessories,1,ed4acab38528488b65a9a9c603ff024a,sao paulo,SP,guarulhos,SP,computers_accessories
29270,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23,4e755f114e50d33b9ac6a56e0d7d3ea9,5,NaN,...,2017-06-27 01:49:04,30b5b5635a79548a48d04162d971848f,sports_leisure,1,f9bbdd976532d50b7816d285a22bd01e,sao paulo,SP,porto alegre,RS,sports_leisure
52486,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19,d055795a562efffefe47ef81e5435322,5,Muito bom,...,2018-07-06 20:30:17,55bfa0307d7a46bed72c492259921231,books_general_interest,1,343e716476e3748b069f980efbaa294e,campinas,SP,ribeirao pires,SP,books_general_interest
72303,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14,e945d1831a3d98008913fc31dcbb804d,5,NaN,...,2017-10-17 10:56:02,44c2baf621113fa7ac95fa06b4afbc68,furniture_decor,1,3f2af2670e104d1bcb54022274daeac5,terra boa,PR,indaiatuba,SP,furniture_decor
76991,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24,ee2d30652e2f7fc00861074f795f5bf0,5,Excelente!,...,2018-07-07 18:48:09,ec165cd31c50585786ffda6feff5d0a6,toys,1,8bdd8e3fd58bafa48af76b2c5fd71974,sao paulo,SP,sao carlos,SP,toys
81004,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18,f48c6c944a5d52dcca8ac5c4ec417cf2,5,NaN,...,2017-12-19 04:15:39,a50acd33ba7a8da8e9db65094fa990a4,auto,1,8581055ce74af1daba164fdbd55a40de,guarulhos,SP,cerquilho,SP,auto
95809,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30,25e11638a3d01a87e8e62338a39eee28,5,NaN,...,2018-07-11 19:27:46,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,1,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,pindamonhangaba,SP,watches_gifts


In [41]:
df_service_eda['order_delivered_customer_date'] = df_service_eda['order_delivered_customer_date'].fillna(
    pd.to_datetime(df_service_eda['order_estimated_delivery_date'])
)

df_service_eda['order_delivered_carrier_date'] = df_service_eda['order_delivered_carrier_date'].fillna(
    pd.to_datetime(df_service_eda['order_approved_at']) + pd.Timedelta(days=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1


In [42]:
df_service_eda.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98665,98665,98665,98665,98665,98665.0,98665,98665,98665.0
mean,2018-01-02 11:40:02,2018-01-02 21:57:29,2018-01-05 16:43:40,2018-01-14 22:37:27,2018-01-26 06:41:08,4.127644,2018-01-14 17:11:58,2018-01-17 20:45:37,1.099255
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.0,2016-10-06 00:00:00,2016-10-07 18:32:28,1.0
25%,2017-09-14 13:57:21,2017-09-14 21:45:17,2017-09-18 19:02:38,2017-09-26 16:48:13,2017-10-05 00:00:00,4.0,2017-09-27 00:00:00,2017-09-29 12:12:07,1.0
50%,2018-01-21 13:28:36,2018-01-22 14:00:56,2018-01-24 18:48:44,2018-02-02 21:39:55,2018-02-16 00:00:00,5.0,2018-02-03 00:00:00,2018-02-06 21:05:12,1.0
75%,2018-05-06 18:49:55,2018-05-07 16:53:20,2018-05-09 10:13:00,2018-05-16 17:29:13,2018-05-28 00:00:00,5.0,2018-05-17 00:00:00,2018-05-20 20:13:32,1.0
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.0,2018-08-31 00:00:00,2018-10-29 12:27:35,20.0
std,NaN,NaN,NaN,NaN,NaN,1.308843,NaN,NaN,0.451793


In [43]:
print("Gesamte Duplikate:", df_service_eda.duplicated().sum())


Gesamte Duplikate: 0


In [44]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'

# 1. Filter auf diese Order_ID
order_data = df_service_eda[(df_service_eda['order_id'] == order_id)]

order_data.head(63).sort_values('order_purchase_timestamp', ascending=True)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
58815,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,5ddab10d5e0a23acb99acf56b62b3276,housewares,1,3d0cd21d41671c46f82cd11176bf7277,joinville,SC,sao paulo,SP,housewares
87107,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,ebf9bc6cd600eadd681384e3116fda85,bed_bath_table,2,822166ed1e47908f7cfb49946d03c726,tres rios,RJ,sao paulo,SP,bed_bath_table


In [45]:
con.execute("CREATE TABLE service_analyse AS SELECT * FROM df_service_eda")